# 07 · EVT 꼬리 사건 시각 점검

06의 **후미_기본·교차_M1, ALL** 층에서 TTC·PET 임계값을 엄격하게 초과한 사건을 점검합니다.
선택 사항으로 후미_기본_2DTTC와 06의 엄격 감도층도 점검할 수 있습니다.
각 층·지표마다 가장 극단적인 K개와 **그 나머지에서** 무작위 K개를 뽑습니다.
지표와 판정의 단위는 사건×지표입니다. 같은 사건이 두 지표에 뽑히면 각각의 극값 시각 그림을 만듭니다.

작업 명세가 인용한 S06(Steinmaßl 2025, PDF p.19)은 사람이 오류·의심·정상으로 구분했고, 두 뒤따름 상황의
극단 TTC 사건 중 16–29%가 자료 오류였습니다. 이 수치를 송도에 적용하지 않습니다.
위치·방향·속력·크기·추적불일치와, 송도의 분리검출·대기행렬·오토바이·차로 라벨 깜빡임을 살펴봅니다.
정사영상은 정적인 배경이며 당시 차량의 사진이 아닙니다. **사고 예측이나 정확도를 검증하는 노트북이 아닙니다.**

코드 셀을 순서대로 사용자가 실행합니다. 2번 설정에서 입력 ID와 K를 정하고,
그림을 본 뒤 `review_sheet.csv`의 **판정·원인·메모**만 작성합니다.
엑셀에서는 **'CSV UTF-8'로 저장**하세요. 전체 입력 정보는 `sample_manifest.csv`에 보존합니다.
마지막 요약 셀은 표를 작성한 후 실행합니다. 판정하지 않은 행은 정상으로 간주하지 않습니다.

## 1. 준비
NumPy·pandas·Pillow를 불러옵니다. 그림은 Pillow로 저장하며, 가우시안 평활의 이산 커널은 NumPy로 계산합니다.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import io
import json
import math
import re
import time
import uuid
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
try:
    from IPython.display import display
except ImportError:
    display = print
FPS = 29.97
FRAME_S = 1 / FPS
NEXT_FRAME_MAX_S = 1.5 * FRAME_S
SIGMA_FRAMES = 14
STATIONARY_MPS = 1 / 3.6
CORNERS = [("tlx", "tly"), ("blx", "bly"), ("brx", "bry"), ("trx", "try")]
ID_COLUMNS = ["drone_id", "leader_id", "follower_id", "vehicle_a_id", "vehicle_b_id", "first_vehicle_id"]
REVIEW_FIELDS = ["판정", "원인", "메모"]
REVIEW_KEYS = {"층": "stratum", "지표": "metric", "선정 방식": "selection"}
print("준비 완료 | pandas", pd.__version__, "| numpy", np.__version__)

준비 완료 | pandas 3.0.5 | numpy 2.4.6


## 2. 입력 ID와 표본 설정
완료된 05·06 ID를 직접 입력합니다. 최신 실행을 자동 선택하지 않으며 실행 중인 05는 읽지 않습니다.
06이 사용한 05·04와 출처가 같은지 확인합니다. 원자료 서명은 해당 01 입력 목록을 통해 확인합니다.
K=30은 각 층·지표·선정 방식별 상한입니다. 남은 사건이 적으면 전부 뽑고 수를 기록합니다.
`THRESHOLD_SET_BY_LAYER = {"후미_기본": "default12", "교차_M1": "stable"}`을 사용합니다.
후미 전체층에는 안정 구간이 없어 M2 PDF p.5의 12%를, 교차에는 M1 PDF p.7 방식의 안정 구간이 있어 stable을 씁니다.
이는 완료된 06의 결과를 선택하는 설정이며 새 임계값을 적합하지 않습니다.
구 단일 설정을 쓰려면 사전을 `None`으로 두고 `THRESHOLD_SET`을 지정합니다. 사전에 없는 선택층도 단일 설정을 사용하고 알립니다.
stable 파일이 없을 때만 default12(또는 구 호환 파일)로
바꾸고 알립니다. `no_stable_region`·`too_few`·`too_few_for_stability` 행은 기본값으로 대신하지 않고 건너뜁니다.
`REVIEW_STRATA`에 `후미_기본_2DTTC`를 추가하면 2D TTC 극값 시각을 씁니다.
이 층의 PET는 후미_기본 PET와 정의가 같습니다. 두 층의 PET 임계값과 꼬리 수가 같을 때만
점검표에 중복 점검으로 표시하며, 임계값이 다르면 구분해 표시합니다.
엄격층은 `후미_기본_엄격`, `교차_M1_엄격`, 전차로 감도층은 `후미_전차로`입니다.
그림 범위는 극값 시각 ±3초입니다. 교차 PET는 첫 통과와 두 번째 통과를 각각 ±3초 패널로 보입니다.
시간 예상은 원자료 크기와 그림 수에 근거한 대략치이며 첫 파일 처리 뒤 실제 속도로 갱신합니다.

In [2]:
SOURCE05_ID = "20260924T125033Z_ffaaff43"  # 완료된 05 실행 ID
SOURCE06_ID = "20260924T191302Z_4491ed18"  # 위 05로 완료한 06 실행 ID
SOURCE04_ID = "20260922T182837Z_4eaba3c7"  # 위 05·06과 연결된 완료 04 실행 ID
THRESHOLD_SET_BY_LAYER = {"후미_기본": "default12", "교차_M1": "stable"}
# 후미 ALL 안정 구간 없음: M2 p.5의 12%; 교차 안정 구간 있음: M1 p.7 방식.
THRESHOLD_SET = "stable"  # 구 단일 설정 호환: 사전이 None이거나 층 키가 없을 때 사용
REVIEW_STRATA = ["후미_기본", "교차_M1"]  # 선택: "후미_기본_2DTTC" 등
K = 30
RANDOM_SEED = 20260924
WINDOW_S = 3.0
EST_READ_MBPS = 25.0  # 예상용 값, 측정된 성능이 아님
EST_FIGURE_S = 2.0
PROJECT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                if (p / "AGENTS.md").is_file() and (p / "data/raw").is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError("프로젝트 또는 analysis 폴더에서 실행하세요.")
RAW = (PROJECT / "data/raw").resolve()
OUTPUT_ROOT = PROJECT / "data/processed/songdo_tail_review"

## 3. 경로와 입력 검증 함수
출력은 이번 `data/processed/songdo_tail_review/RUN_ID/` 내부로 제한하고 입력 경로와의 겹침을 막습니다.
CSV·설정은 읽기 전후 크기·수정 시각과 해시를 기록합니다. 필요한 원자료는 01의 크기·수정 시각과 대조합니다.
04의 요약 표에는 아핀 계수가 없으므로 같은 실행의 `progress.csv` 계수를 연결합니다.

In [3]:
def now_utc():
    return datetime.now(timezone.utc).isoformat()


def signature(path):
    stat = Path(path).stat()
    return {"size_bytes": str(stat.st_size), "mtime_ns": str(stat.st_mtime_ns)}


def assert_signature(path, expected):
    if signature(path) != {k: str(expected[k]) for k in ["size_bytes", "mtime_ns"]}:
        raise RuntimeError("입력 크기·수정 시각 변경: " + str(path))


def within(path, root):
    path, root = Path(path).resolve(), Path(root).resolve()
    if not path.is_relative_to(root) or path == root:
        raise ValueError("허용 경로 밖: " + str(path))
    return path


def run_input(folder, run_id):
    if not isinstance(run_id, str) or not re.fullmatch(r"[A-Za-z0-9_-]+", run_id):
        raise ValueError("완료된 실행 ID를 입력하세요: " + str(run_id))
    return within(PROJECT / "data/processed" / folder / run_id, PROJECT / "data/processed" / folder)


def target_path(path):
    target = within(path, RUN_DIR)
    within(RUN_DIR, OUTPUT_ROOT)
    within(OUTPUT_ROOT, PROJECT / "data/processed")
    if Path(OUTPUT_ROOT).resolve() != (PROJECT / "data/processed/songdo_tail_review").resolve():
        raise ValueError("출력 상위 폴더는 songdo_tail_review여야 합니다.")
    for source in [RAW, SOURCE01, SOURCE04, SOURCE05, SOURCE06]:
        if target.is_relative_to(Path(source).resolve()):
            raise ValueError("입력 폴더에는 저장할 수 없습니다.")
    return target


def save_csv(table, path):
    target = target_path(path)
    temporary = target_path(target.with_name(target.name + ".tmp"))
    table.to_csv(temporary, index=False, encoding="utf-8-sig")
    temporary.replace(target)


def save_json(value, path):
    target = target_path(path)
    temporary = target_path(target.with_name(target.name + ".tmp"))
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    temporary.replace(target)


controls = {}
def read_control(path, **kwargs):
    path = within(path, PROJECT / "data/processed")
    before = signature(path)
    content = path.read_bytes()
    assert_signature(path, before)
    controls[path.relative_to(PROJECT).as_posix()] = {
        **before, "sha256": hashlib.sha256(content).hexdigest()}
    if path.suffix == ".json":
        return json.loads(content.decode("utf-8-sig"))
    return pd.read_csv(io.BytesIO(content), **kwargs)


def read_review_sheet(path):
    try:
        sheet = read_control(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
        encoding = "utf-8-sig"
    except UnicodeDecodeError:
        sheet = read_control(path, dtype=str, keep_default_na=False, encoding="cp949")
        encoding = "cp949"
    print("점검표 읽기 인코딩:", encoding)
    if encoding == "cp949":
        print("다음에는 CSV UTF-8로 저장하세요.")
    blank_rows = sheet.apply(lambda column: column.fillna("").astype(str).str.strip().eq("")).all(axis=1)
    n_blank = int(blank_rows.sum())
    if n_blank:
        print(f"빈 행 {n_blank}개를 제외했습니다")
    return sheet.loc[~blank_rows].reset_index(drop=True)


def completed_meta(path, run_id):
    meta = read_control(path / "run_metadata.json")
    if meta.get("status") != "completed" or meta.get("run_id") != run_id:
        raise ValueError("완료된 실행이 아닙니다: " + str(path))
    return meta


def load_thresholds(source, requested):
    if requested not in {"stable", "default12"}:
        raise ValueError("THRESHOLD_SET은 stable 또는 default12여야 합니다.")
    source = Path(source)
    effective = requested
    chosen = source / f"univariate_gpd_{requested}.csv"
    if requested == "stable" and not chosen.is_file():
        effective = "default12"
        chosen = source / "univariate_gpd_default12.csv"
        print("알림: stable 임계값 파일이 없어 default12로 전환합니다.")
    if effective == "default12" and not chosen.is_file():
        chosen = source / "univariate_gpd.csv"
        print("알림: 구 06의 univariate_gpd.csv를 default12로 읽습니다.")
    table = read_control(chosen)
    required = {"유형", "지점", "지표", "threshold"}
    if not required.issubset(table.columns):
        raise ValueError("06 임계값 표의 필수 열이 없습니다: " + str(required - set(table.columns)))
    provenance = {"requested": requested, "effective": effective, "file": chosen.name,
                  "fallback": effective != requested}
    print("사용 임계값:", effective, "|", chosen.name)
    return table, provenance


def load_inputs():
    global SOURCE01, SOURCE04, SOURCE05, SOURCE06, threshold_selection, literature_procedure, literature_audit
    if SOURCE05_ID in {"20260923T102351Z_875f4ea7", "20260924T111836Z_ce5dd4cf"}:
        raise ValueError("프로젝트에서 사용 금지한 05 실행입니다.")
    SOURCE05 = run_input("songdo_events", SOURCE05_ID)
    SOURCE06 = run_input("songdo_evt", SOURCE06_ID)
    SOURCE04 = run_input("songdo_structure", SOURCE04_ID)
    m05 = completed_meta(SOURCE05, SOURCE05_ID)
    m06 = completed_meta(SOURCE06, SOURCE06_ID)
    m04 = completed_meta(SOURCE04, SOURCE04_ID)
    if (m05.get("source04_run_id") != SOURCE04_ID
            or m06.get("source04_run_id") != SOURCE04_ID
            or m06.get("source05_run_id") != SOURCE05_ID
            or m05.get("source01_run_id") != m04.get("source01_run_id")):
        raise ValueError("04·05·06의 입력 실행 연결이 다릅니다.")
    SOURCE01 = run_input("songdo_movement", m04["source01_run_id"])
    completed_meta(SOURCE01, m04["source01_run_id"])
    manifest = read_control(SOURCE01 / "input_manifest.csv", dtype="string")
    if manifest["source_file"].duplicated().any():
        raise ValueError("01 입력 목록에 중복 파일이 있습니다.")
    hand = read_control(SOURCE04 / "coordinate_handedness_by_file.csv")
    progress = read_control(SOURCE04 / "progress.csv")
    coeff = ["hand_a11", "hand_a12", "hand_a21", "hand_a22", "hand_b1", "hand_b2"]
    affines = hand.merge(progress[["source_file", "status"] + coeff], on="source_file",
                         how="left", validate="one_to_one")
    rear = read_control(SOURCE05 / "event_catalog_rear_end.csv", dtype={c: "string" for c in ID_COLUMNS})
    cross = read_control(SOURCE05 / "event_catalog_crossing.csv", dtype={c: "string" for c in ID_COLUMNS})
    thresholds, threshold_selection = load_layer_thresholds(SOURCE06, REVIEW_STRATA)
    literature_procedure = m06.get("literature_procedure")
    if "literature_procedure" in m06 and not isinstance(literature_procedure, dict):
        raise ValueError("06의 literature_procedure 기록이 유효하지 않습니다.")
    # 선별 이전 카탈로그의 1-based 행을 유지해 기존 07 사건 판정을 연결한다.
    rear["_source_catalog_row"] = np.arange(1, len(rear) + 1)
    cross["_source_catalog_row"] = np.arange(1, len(cross) + 1)
    rear, cross, literature_audit = apply_literature_procedure(rear, cross, literature_procedure)
    return rear, cross, thresholds, manifest, affines


def load_layer_thresholds(source, strata_names):
    mapping = globals().get("THRESHOLD_SET_BY_LAYER")
    if mapping is None:
        return load_thresholds(source, THRESHOLD_SET)
    if not isinstance(mapping, dict) or any(v not in {"stable", "default12"} for v in mapping.values()):
        raise ValueError("THRESHOLD_SET_BY_LAYER는 층: stable/default12 사전 또는 None이어야 합니다.")
    tables, cache, provenance = [], {}, {}
    for name in strata_names:
        requested = mapping.get(name, THRESHOLD_SET)
        if name not in mapping:
            print(f"알림: THRESHOLD_SET_BY_LAYER[{name!r}] 없음 → THRESHOLD_SET={requested!r} 사용")
        if requested not in cache:
            cache[requested] = load_thresholds(source, requested)
        table, choice = cache[requested]
        tables.append(table.loc[table["유형"].eq(name)].assign(threshold_set=choice["effective"]))
        provenance[name] = dict(choice)
    if not tables:
        raise ValueError("REVIEW_STRATA는 비어 있지 않아야 합니다.")
    return pd.concat(tables, ignore_index=True), {"by_layer": provenance}


def review_config():
    mapping = globals().get("THRESHOLD_SET_BY_LAYER")
    return (SOURCE04_ID, SOURCE05_ID, SOURCE06_ID, K, RANDOM_SEED, THRESHOLD_SET,
            tuple(REVIEW_STRATA), None if mapping is None else tuple(sorted(mapping.items())))

## 4. 06과 같은 부호 반전 및 표본 함수
아래 `as_negated`는 fix_0506 v2의 06 빌더 함수를 그대로 복사한 것입니다.
`literature_prescreen`과 `spatial_*` 7개는 lit_procedure v2의 06에서 텍스트 그대로 복사했습니다.
06 메타데이터의 선별 초·유턴 설정·공간 버전·지도 ID를 읽고 함수 및 기하 입력 SHA256을 대조합니다.
공간 기준점은 선별 전 전체 M1에서 계산하고, 실제 제외는 **사전선별 → 유턴 → 공간** 순서로 적용합니다.
문헌 절차가 없는 구형 06은 알림 후 기존 방식으로 처리합니다. 절차 기록이 있는데 불완전하거나 다르면 멈춥니다.
공간 재현에는 SciPy·Matplotlib이 필요합니다. 공간 함수 중 `spatial_save`는 복사만 하며 호출하지 않습니다.
기본 후미는 `through_or_shared`, 교차는 `m1_left_turn_opposing_through`에 적용합니다.
TTC와 PET별 상태·떨림·차로변경 제외도 동일합니다. `position_jitter` OR로 일괄 제외하지 않습니다.
엄격층은 오토바이와 후미 지표별 뒤차 속력을, 2D 층은 별도 2D 상태·떨림·유형을 사용합니다.
예를 들어 교차에서 TTC가 차체 겹침이어도 PET가 computed라면 06처럼 PET는 남을 수 있습니다.
유한한 `−값 > threshold`만 꼬리입니다. 동률은 카탈로그 원래 행 번호로 순서를 정합니다.

05에는 사건 ID가 없어 **05 실행 ID:유형:카탈로그 데이터 행 번호(1부터)**를 새 식별자로 보존합니다.
점검 행 ID에는 층과 지표도 포함합니다. 서로 다른 층의 같은 사건을 독립적으로 기록합니다.
선정은 층·지표별 고정 시드로 비복원 추출하며, 임계값 부재는 건너뛰는 이유를 기록합니다.

In [4]:
def as_negated(table, ttc_col, pet_col, strict=False, speed_cols=None, ttc_status="ttc_status", types=None,
               ttc_jitter="jitter_ttc"):
    ttc = pd.to_numeric(table[ttc_col], errors="coerce")
    pet = pd.to_numeric(table[pet_col], errors="coerce")
    jitter_x_removed = table[ttc_status].eq("computed") & np.isfinite(ttc) & table[ttc_jitter]
    jitter_y_removed = table["pet_status"].eq("computed") & np.isfinite(pet) & table["jitter_pet"]
    eligible_x = ~table[ttc_jitter]
    eligible_y = ~table["jitter_pet"]
    ok_x = table[ttc_status].eq("computed") & eligible_x
    ok_y = table["pet_status"].eq("computed") & eligible_y
    lane_change = pd.Series(False, index=table.index)
    if types is not None:
        lc_x = table[types[0]].eq("lane_change")
        lc_y = table[types[1]].eq("lane_change")
        lane_change = (lc_x & ok_x) | (lc_y & ok_y)
        ok_x &= ~lc_x
        ok_y &= ~lc_y
        eligible_x &= ~lc_x
        eligible_y &= ~lc_y
    base_x, base_y = ok_x.copy(), ok_y.copy()
    moto = table["motorcycle_involved"]
    stopped_x = stopped_y = pd.Series(False, index=table.index)
    if strict:
        ok_x &= ~moto
        ok_y &= ~moto
        eligible_x &= ~moto
        eligible_y &= ~moto
        if speed_cols is not None:
            stopped_x = pd.to_numeric(table[speed_cols[0]], errors="coerce") < STATIONARY_MPS
            stopped_y = pd.to_numeric(table[speed_cols[1]], errors="coerce") < STATIONARY_MPS
            ok_x &= ~stopped_x
            ok_y &= ~stopped_y
            eligible_x &= ~stopped_x
            eligible_y &= ~stopped_y
    x = np.where(ok_x, -ttc, np.nan)
    y = np.where(ok_y, -pet, np.nan)
    ttc_overlap = table[ttc_status].eq("body_overlap")
    pet_overlap = table["pet_status"].eq("body_overlap")
    usable_before = base_x | base_y
    return pd.DataFrame({"site": table["site"].to_numpy(), "x": x, "y": y,
                         "body_overlap": (ttc_overlap | pet_overlap).to_numpy(),
                         "x_overlap": np.where(ttc_overlap & eligible_x, -ttc, np.nan),
                         "y_overlap": np.where(pet_overlap & eligible_y, -pet, np.nan),
                         "jitter_ttc_removed": jitter_x_removed.to_numpy(),
                         "jitter_pet_removed": jitter_y_removed.to_numpy(),
                         "lane_change_removed": lane_change.to_numpy(),
                         "moto_removed": (moto & strict & usable_before).to_numpy(),
                         "stopped_removed": (((stopped_x & base_x) | (stopped_y & base_y)) & ~(moto & strict)).to_numpy()})

NEGATION_SHA256 = 'c797367bc0741e927ba5eefb338f3d22314d0e2161326b47349e7508d8236d91'

SOURCE06_FUNCTION_SHA256 = {'literature_prescreen': '08000d82d5f741c186d7fbaec7b7e2f220ab728c7ac2717f408ba4ead780c060', 'spatial_affine': 'e8c8741f598b653da41200d1219010f8f11753d716df6a2cd15b969bff6a3212', 'spatial_centre_polygon': 'df1d35c90810ef7499e20c070cef9e7e27d2770a7bd77186021325639dcc43ec', 'spatial_classify': 'cd4f69f0e288f8c119a526c6adaefa8bec2a24fbc0172660af87378c0823563b', 'spatial_boundary_distance': '34055d016d4b87479d9b1ad5fd50289743aa6da79083b6618e302032d216a4c8', 'spatial_prepare': 'ad3475b82567ebe3d1893b0f96e45933b64cf8a60fe06ce0cbc807d70947102b', 'spatial_filter': '9df7ba9ae91d14b88658dbb5fe4db6909252ac64aace36257100307f451e4463', 'spatial_save': '90f46864de0e00ae3b6cad0a7ed05a2aafee89e8e9a536b2ca90c04a793ddc93'}

def literature_prescreen(rear, cross):
    """사건 단위 4초 OR 선별 후 M1 유턴만 제외; 입력과 지표 값은 보존한다."""
    if (isinstance(PRESCREEN_SECONDS, (bool, np.bool_))
            or not np.isfinite(PRESCREEN_SECONDS) or PRESCREEN_SECONDS <= 0):
        raise ValueError("PRESCREEN_SECONDS는 양의 유한한 초 값이어야 합니다.")
    if not isinstance(EXCLUDE_UTURN_M1, (bool, np.bool_)):
        raise ValueError("EXCLUDE_UTURN_M1은 True/False여야 합니다.")

    def selected(table, pet_column):
        ttc = pd.to_numeric(table["min_ttc_s"], errors="coerce")
        pet = pd.to_numeric(table[pet_column], errors="coerce")
        return ((np.isfinite(ttc) & ttc.lt(PRESCREEN_SECONDS))
                | (np.isfinite(pet) & pet.lt(PRESCREEN_SECONDS))).fillna(False)

    rear_keep = selected(rear, "min_pet_s")
    cross_keep = selected(cross, "pet_s")
    m1 = cross["m1_left_turn_opposing_through"].astype(str).str.lower().eq("true")
    uturn = pd.Series(False, index=cross.index)
    invalid_flows = pd.Series(0, index=cross.index, dtype="int64")
    for column in ("flow_a", "flow_b"):
        flow = (cross[column] if column in cross else pd.Series(pd.NA, index=cross.index))
        parts = flow.astype("string").str.extract(r"^\s*([0-9]+)_[0-9]+\s*→\s*([0-9]+)_[0-9]+\s*$")
        valid = parts.notna().all(axis=1)
        same = parts[0].str.lstrip("0").eq(parts[1].str.lstrip("0")).fillna(False)
        uturn |= valid & same
        invalid_flows += (~valid).astype("int64")
    removed_uturn = cross_keep & m1 & uturn & EXCLUDE_UTURN_M1
    rows = []
    groups = [
        (["후미_기본", "후미_기본_엄격", "후미_기본_2DTTC"], rear,
         rear["lane_role"].eq("through_or_shared"), rear_keep, False),
        (["후미_전차로"], rear, pd.Series(True, index=rear.index), rear_keep, False),
        (["교차_M1", "교차_M1_엄격"], cross, m1, cross_keep, True),
    ]
    for names, table, membership, keep, is_cross in groups:
        for site in ["ALL", *sorted(table.loc[membership, "site"].dropna().unique())]:
            before = membership if site == "ALL" else membership & table["site"].eq(site)
            after_screen = before & keep
            removed = int((after_screen & removed_uturn).sum()) if is_cross else 0
            audit = {"지점": site, "선별전_사건": int(before.sum()),
                     "사전선별_제외": int((before & ~keep).sum()),
                     "사전선별후_사건": int(after_screen.sum()), "유턴_제외": removed,
                     "유턴판정불가_사건": int((after_screen & invalid_flows.gt(0)).sum()) if is_cross else 0,
                     "유턴판정불가_흐름": int(invalid_flows.loc[after_screen].sum()) if is_cross else 0,
                     "선별후_사건": int(after_screen.sum()) - removed}
            rows.extend({"유형": name, **audit} for name in names)
    return rear.loc[rear_keep].copy(), cross.loc[cross_keep & ~removed_uturn].copy(), pd.DataFrame(rows)


def spatial_affine(row):
    """07과 같은 EPSG 미터 = A @ Ortho 픽셀 + b 검사와 변환 계약."""
    a = np.array([[row["hand_a11"], row["hand_a12"]], [row["hand_a21"], row["hand_a22"]]], float)
    b = np.array([row["hand_b1"], row["hand_b2"]], float)
    if not np.isfinite(a).all() or not np.isfinite(b).all() or np.linalg.cond(a) > 1e8:
        raise ValueError("유효한 04 아핀 계수가 없습니다.")
    if row["status"] != "completed" or row["hand_local_frame"] != "right_handed":
        raise ValueError("04 완료 상태 또는 EPSG 오른손 좌표계가 확인되지 않았습니다.")
    if not np.isclose(np.linalg.det(a), float(row["hand_det"]), rtol=1e-5, atol=1e-12):
        raise ValueError("04 요약표와 아핀 계수의 행렬식이 다릅니다.")
    return a, b


def spatial_centre_polygon(lanes, anchor):
    """공식 차로의 정지선 쪽 짧은 변 → 접근로별 최근접 Section → 볼록 껍질.

    본 연구의 조작적 정의이며 이 구성 절차 자체의 문헌 근거는 not_found.
    polygon_valid=False도 좌표가 유한하면 사용한다(10 MD 144행).
    """
    from scipy.spatial import ConvexHull, QhullError
    columns = ["tlx", "tly", "blx", "bly", "brx", "bry", "trx", "try"]
    quads, by_section = [], {}
    for row in lanes.to_dict("records"):
        q = np.asarray([row[c] for c in columns], float).reshape(4, 2)
        if not np.isfinite(q).all():
            continue
        quads.append(q)
        if not np.isfinite(anchor).all():
            continue
        section = str(row["Section"])
        approach = section.split("_")[0]
        if not approach.isdigit():
            raise ValueError("접근로 번호를 해석할 수 없는 Section: " + section)
        edges = [(q[i], q[(i + 1) % 4]) for i in range(4)]
        edges.sort(key=lambda e: np.linalg.norm(e[0] - e[1]))
        edge = min(edges[:2], key=lambda e: np.linalg.norm((e[0] + e[1]) / 2 - anchor))
        distance = float(np.linalg.norm((edge[0] + edge[1]) / 2 - anchor))
        by_section.setdefault(section, []).append((distance, edge, str(row["Lane"])))
    best = {}
    for section, edges in by_section.items():
        approach = section.split("_")[0]
        distance = min(e[0] for e in edges)
        if approach not in best or distance < best[approach][0]:
            best[approach] = (distance, section)
    endpoints, used = [], []
    for approach, (distance, section) in best.items():
        for _, edge, lane in by_section[section]:
            endpoints.extend(edge)
            used.append({"approach": approach, "Section": section, "Lane": lane,
                         "section_min_distance_px": distance,
                         "edge_x1": float(edge[0][0]), "edge_y1": float(edge[0][1]),
                         "edge_x2": float(edge[1][0]), "edge_y2": float(edge[1][1])})
    points = np.asarray(endpoints, float).reshape(-1, 2)
    hull, status = np.empty((0, 2)), "insufficient_centre_vertices"
    if len(points) >= 3:
        try:
            hull = points[ConvexHull(points).vertices]
            status = "ok"
        except QhullError:
            status = "degenerate_centre_polygon"
    if not np.isfinite(anchor).all():
        status = "no_finite_m1_anchor"
    return {"quads": quads, "hull": hull, "sections": used, "status": status,
            "anchor": np.asarray(anchor, float)}


def spatial_classify(xy, geometry):
    """최신 SPEC: on_lane 우선, 다음 centre, 나머지 outside_centre. 경계 띠로 변경하지 않음."""
    from matplotlib.path import Path as PolygonPath
    xy = np.asarray(xy, float).reshape(-1, 2)
    result = np.full(len(xy), "invalid_conflict_point", dtype=object)
    valid = np.isfinite(xy).all(axis=1)
    if geometry["status"] != "ok":
        result[valid] = geometry["status"]
        return result
    on_lane = np.zeros(len(xy), bool)
    for quad in geometry["quads"]:
        on_lane |= PolygonPath(quad).contains_points(xy)
    in_centre = PolygonPath(geometry["hull"]).contains_points(xy)
    result[valid] = np.where(on_lane[valid], "on_lane", np.where(in_centre[valid], "centre", "outside_centre"))
    return result


def spatial_boundary_distance(xy, polygons, affine):
    """점과 다각형 변을 같은 04 선형변환으로 미터화. 0.5m는 표시 전용."""
    xy = np.asarray(xy, float).reshape(-1, 2) @ affine.T
    distance = np.full(len(xy), np.inf)
    for polygon in polygons:
        polygon = np.asarray(polygon, float) @ affine.T
        for first, last in zip(polygon, np.roll(polygon, -1, axis=0)):
            delta = last - first
            length2 = float(delta @ delta)
            t = np.zeros(len(xy)) if length2 == 0 else np.clip((xy - first) @ delta / length2, 0, 1)
            distance = np.minimum(distance, np.linalg.norm(xy - (first + t[:, None] * delta), axis=1))
    distance[~np.isfinite(distance)] = np.nan
    return distance


def spatial_prepare(cross, source04, geometry_dir):
    """사전선별 전 전체 M1의 지점별 중앙값을 방향 기준으로 고정한다(시험 원형과 동일)."""
    from pathlib import Path
    import hashlib
    source04, geometry_dir = Path(source04), Path(geometry_dir)
    paths = {"handedness": source04 / "coordinate_handedness_by_file.csv",
             "affine_progress": source04 / "progress.csv", "map_index": geometry_dir / "map_index.csv"}
    provenance = {key: {"path": str(path), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
                  for key, path in paths.items()}
    hand, progress, lanes = (pd.read_csv(paths[key]) for key in ("handedness", "affine_progress", "map_index"))
    coefficients = ["hand_a11", "hand_a12", "hand_a21", "hand_a22", "hand_b1", "hand_b2"]
    affines = hand.merge(progress[["source_file", "status"] + coefficients], on="source_file",
                         how="left", validate="one_to_one")
    if affines["file_stem"].duplicated().any():
        raise ValueError("04 아핀 file_stem 중복")
    affines = affines.set_index("file_stem")
    required = {"site", "file_stem", "conflict_x", "conflict_y", "m1_left_turn_opposing_through"}
    if not required.issubset(cross.columns):
        raise ValueError("공간 조건 입력 열 누락: " + str(sorted(required - set(cross.columns))))
    m1 = cross["m1_left_turn_opposing_through"].astype(str).str.lower().eq("true").to_numpy()
    records = cross.loc[m1].copy().reset_index(drop=True)
    records.insert(0, "catalog_row", np.flatnonzero(m1) + 1)  # 07 manifest와 같은 1-based 행
    xy = np.full((len(records), 2), np.nan)
    matrices = {}
    for stem, indices in records.groupby("file_stem", dropna=False).groups.items():
        if stem not in affines.index:
            raise ValueError("04 아핀 파일 미확인: " + str(stem))
        row = affines.loc[stem]
        if not records.loc[indices, "site"].eq(row["site"]).all():
            raise ValueError("05 사건과 04 아핀의 지점 불일치: " + str(stem))
        a, b = spatial_affine(row)
        matrices[stem] = a
        local = records.loc[indices, ["conflict_x", "conflict_y"]].apply(pd.to_numeric, errors="coerce").to_numpy(float)
        xy[indices] = np.linalg.solve(a, (local - b).T).T
    records["ortho_x"], records["ortho_y"] = xy[:, 0], xy[:, 1]
    records["spatial_class"] = "unclassified"
    records["boundary_distance_m"] = np.nan
    geometries, section_rows, vertex_rows, site_rows = {}, [], [], []
    sites = sorted(set(lanes["site"].dropna()) | set(records["site"].dropna()))
    for site in sites:
        indices = records.index[records["site"].eq(site)]
        points = xy[indices]
        finite = points[np.isfinite(points).all(axis=1)]
        anchor = np.median(finite, axis=0) if len(finite) else np.array([np.nan, np.nan])
        geometry = spatial_centre_polygon(lanes.loc[lanes["site"].eq(site)], anchor)
        geometries[site] = geometry
        records.loc[indices, "spatial_class"] = spatial_classify(points, geometry)
        for stem, group in records.loc[indices].groupby("file_stem"):
            records.loc[group.index, "boundary_distance_m"] = spatial_boundary_distance(
                xy[group.index], [*geometry["quads"], geometry["hull"]], matrices[stem])
        section_rows.extend({"site": site, **row} for row in geometry["sections"])
        vertex_rows.extend({"site": site, "vertex": i, "ortho_x": float(p[0]), "ortho_y": float(p[1])}
                           for i, p in enumerate(geometry["hull"]))
        counts = records.loc[indices, "spatial_class"].value_counts()
        site_rows.append({"site": site, "geometry_status": geometry["status"], "all_m1_events": len(indices),
                          "centre": int(counts.get("centre", 0)), "on_lane": int(counts.get("on_lane", 0)),
                          "outside_centre": int(counts.get("outside_centre", 0)),
                          "unclassified": int((~records.loc[indices, "spatial_class"].isin(["centre", "on_lane", "outside_centre"])).sum()),
                          "anchor_x": anchor[0], "anchor_y": anchor[1],
                          "approach_count": len({row["approach"] for row in geometry["sections"]}),
                          "selected_sections": ";".join(dict.fromkeys(row["Section"] for row in geometry["sections"])),
                          "finite_lane_polygons": len(geometry["quads"])})
    # 표시용 0.5m 띠는 분류·잔존 여부에 사용하지 않는다(최신 v2 보완: centre만 잔존).
    records["boundary_near_05m"] = records["boundary_distance_m"].le(0.5)
    annotated = cross.copy()
    classes = np.full(len(cross), "not_m1", dtype=object)
    near = np.zeros(len(cross), bool)
    classes[m1] = records["spatial_class"].to_numpy()
    near[m1] = records["boundary_near_05m"].to_numpy()
    annotated["_m1_spatial_class"], annotated["_m1_boundary_near_05m"] = classes, near
    return annotated, {"events": records, "geometries": geometries,
                       "sections": pd.DataFrame(section_rows), "vertices": pd.DataFrame(vertex_rows),
                       "sites": pd.DataFrame(site_rows), "inputs": provenance}


def spatial_filter(cross, audit):
    """사전선별→유턴 제외 뒤의 M1에만 공간 조건을 적용하고 사건 수를 감사한다."""
    m1 = cross["m1_left_turn_opposing_through"].astype(str).str.lower().eq("true")
    classes = cross["_m1_spatial_class"]
    bad = m1 & ~classes.isin(["centre", "on_lane", "outside_centre"])
    if bad.any():
        raise ValueError("남은 M1 사건의 공간 판정 불가: " + str(classes.loc[bad].value_counts().to_dict()))
    audit = audit.copy()
    audit["유턴제외후_사건"] = audit["선별후_사건"]
    for column in ["공간_on_lane_제외", "공간_outside_centre_제외", "공간_경계근접_표시"]:
        audit[column] = 0
    for index, row in audit.loc[audit["유형"].isin(["교차_M1", "교차_M1_엄격"])].iterrows():
        membership = m1 if row["지점"] == "ALL" else m1 & cross["site"].eq(row["지점"])
        if int(membership.sum()) != row["유턴제외후_사건"]:
            raise ValueError("유턴 제외 뒤 사건 감사표 불일치")
        for label in ["on_lane", "outside_centre"]:
            audit.loc[index, "공간_" + label + "_제외"] = int((membership & classes.eq(label)).sum())
        audit.loc[index, "공간_경계근접_표시"] = int((membership & cross["_m1_boundary_near_05m"]).sum())
        audit.loc[index, "선별후_사건"] = int((membership & classes.eq("centre")).sum())
    return cross.loc[~m1 | classes.eq("centre")].copy(), audit


def spatial_save(context, project, run_dir):
    """중앙 폴리곤·모든 선별 전 M1 분류 점을 공식 정사영상에 표시한다."""
    from pathlib import Path
    from PIL import Image, ImageDraw
    import hashlib
    out = Path(run_dir).resolve() / "intersection_geometry"
    out.mkdir(parents=True, exist_ok=False)
    for name in ["events", "sections", "vertices", "sites"]:
        context[name].to_csv(out / (name + ".csv"), index=False, encoding="utf-8-sig")
    figures = []
    colors = {"centre": "lime", "on_lane": "magenta", "outside_centre": "orange"}
    for site, geometry in context["geometries"].items():
        source = Path(project) / "data/raw/orthophotos/orthophotos" / (str(site) + ".png")
        if not source.is_file():
            raise FileNotFoundError("공식 Ortho 정사영상 없음: " + str(source))
        with Image.open(source) as original:
            original_size = original.size
            picture = original.convert("RGB")
            picture.thumbnail((1600, 1600))  # 표시 해상도이며 분석 수치 기준 아님.
        sx, sy = picture.width / original_size[0], picture.height / original_size[1]
        draw = ImageDraw.Draw(picture)
        def display_points(points):
            return [(float(p[0]) * sx, float(p[1]) * sy) for p in points]
        for quad in geometry["quads"]:
            draw.polygon(display_points(quad), outline="cyan", width=1)
        if len(geometry["hull"]):
            draw.polygon(display_points(geometry["hull"]), outline="white", width=3)
        for row in geometry["sections"]:
            draw.line(display_points([[row["edge_x1"], row["edge_y1"]], [row["edge_x2"], row["edge_y2"]]]), fill="yellow", width=4)
        events = context["events"].loc[context["events"]["site"].eq(site)]
        for row in events.to_dict("records"):
            if not np.isfinite([row["ortho_x"], row["ortho_y"]]).all():
                continue
            x, y = row["ortho_x"] * sx, row["ortho_y"] * sy
            draw.ellipse([x-2, y-2, x+2, y+2], fill=colors.get(row["spatial_class"], "red"))
        count = events["spatial_class"].value_counts()
        header = (f"{site} | all M1 before filters: {len(events)} | " + geometry["status"] + "\n"
                  + " | ".join(f"{key}: {int(count.get(key, 0))}" for key in colors)
                  + "\nwhite: centre hull; yellow: selected edges; cyan: lanes"
                  + "\ngreen: centre; magenta: on_lane; orange: outside_centre"
                  + "\nsections: " + ", ".join(dict.fromkeys(row["Section"] for row in geometry["sections"])))
        draw.rectangle((0, 0, picture.width, 85), fill="black")
        draw.text((8, 5), header, fill="white", spacing=3)
        name = str(site) + "_intersection.png"
        picture.save(out / name)
        figures.append({"site": site, "figure": name, "orthophoto": str(source),
                        "orthophoto_sha256": hashlib.sha256(source.read_bytes()).hexdigest(),
                        "original_width": original_size[0], "original_height": original_size[1]})
    pd.DataFrame(figures).to_csv(out / "figures.csv", index=False, encoding="utf-8-sig")
    return figures


def apply_literature_procedure(rear, cross, procedure):
    global PRESCREEN_SECONDS, EXCLUDE_UTURN_M1
    if procedure is None:
        print("알림: 구형 06에 literature_procedure가 없어 문헌 사전선별·유턴·공간 조건 없이 기존 방식으로 처리합니다.")
        return rear, cross, pd.DataFrame()
    settings = procedure.get("settings") if isinstance(procedure, dict) else None
    if not isinstance(settings, dict):
        raise ValueError("06의 문헌 사전선별 설정이 없습니다.")
    seconds, exclude = settings.get("PRESCREEN_SECONDS"), settings.get("EXCLUDE_UTURN_M1")
    if (isinstance(seconds, bool) or not isinstance(seconds, (int, float))
            or not np.isfinite(seconds) or seconds <= 0 or not isinstance(exclude, bool)):
        raise ValueError("06의 문헌 사전선별 설정이 유효하지 않습니다.")
    if procedure.get("function_sha256") != SOURCE06_FUNCTION_SHA256["literature_prescreen"]:
        raise ValueError("06·07 literature_prescreen 함수 SHA256이 다릅니다.")
    spatial = procedure.get("spatial")
    if not isinstance(spatial, dict) or spatial.get("version") != "stop_line_hull_v2":
        raise ValueError("06에 stop_line_hull_v2 공간 절차 기록이 없습니다.")
    expected = {n: sha for n, sha in SOURCE06_FUNCTION_SHA256.items() if n.startswith("spatial_")}
    if spatial.get("function_sha256") != expected:
        raise ValueError("06·07 공간 함수 SHA256이 다릅니다.")
    if (spatial.get("order") != ["prescreen", "uturn", "spatial_centre_only"]
            or spatial.get("boundary_affects_selection") is not False
            or spatial.get("boundary_display_m") != 0.5):
        raise ValueError("06 공간 제외 순서 또는 경계 표시 계약이 다릅니다.")
    geometry_dir = run_input("songdo_route_geometry", spatial.get("geometry_id"))
    paths = {"handedness": SOURCE04 / "coordinate_handedness_by_file.csv",
             "affine_progress": SOURCE04 / "progress.csv", "map_index": geometry_dir / "map_index.csv"}
    expected_inputs = spatial.get("input_provenance")
    if not isinstance(expected_inputs, dict) or set(expected_inputs) != set(paths):
        raise ValueError("06 공간 입력 SHA256 기록이 없습니다.")
    for key, path in paths.items():
        read_control(path)  # 입력 서명과 SHA256을 07 실행 기록에 보존한다.
        if controls[path.relative_to(PROJECT).as_posix()]["sha256"] != expected_inputs[key].get("sha256"):
            raise ValueError("06 이후 공간 지도 또는 04 아핀 입력이 바뀌었습니다: " + key)
    # 중앙값 anchor는 필터 전 전체 M1에서 구한다. 이 단계에서는 사건을 제외하지 않는다.
    cross, context = spatial_prepare(cross, SOURCE04, geometry_dir)
    for key, path in paths.items():
        assert_signature(path, controls[path.relative_to(PROJECT).as_posix()])
        if context["inputs"][key]["sha256"] != expected_inputs[key]["sha256"]:
            raise ValueError("공간 준비 중 입력 SHA256이 바뀌었습니다: " + key)
    PRESCREEN_SECONDS, EXCLUDE_UTURN_M1 = float(seconds), exclude
    rear, cross, audit = literature_prescreen(rear, cross)
    cross, audit = spatial_filter(cross, audit)
    print(f"06 문헌 절차 적용: {seconds:g}초 OR → 유턴 제외={exclude} → {spatial['version']} | 지도 {spatial['geometry_id']}")
    return rear, cross, audit


def prepare_catalog(table, kind, source05_id):
    table = table.copy().reset_index(drop=True)
    table["catalog_row"] = (table.pop("_source_catalog_row") if "_source_catalog_row" in table
                            else np.arange(1, len(table) + 1))
    table["catalog_kind"] = kind
    table["event_id"] = pd.Series([f"{source05_id}:{kind}:{i:08d}" for i in table["catalog_row"]], dtype="string")
    for column in [c for c in ["position_jitter", "jitter_ttc", "jitter_pet", "jitter_ttc_2d", "motorcycle_involved"] if c in table]:
        table[column] = table[column].astype(str).str.lower().eq("true")
    return table


def make_strata(rear, cross, strata_names):
    if not strata_names or len(set(strata_names)) != len(strata_names):
        raise ValueError("REVIEW_STRATA는 중복 없는 비어 있지 않은 층 목록이어야 합니다.")
    result = {}
    for name in strata_names:
        if name in {"후미_기본", "후미_기본_엄격", "후미_기본_2DTTC", "후미_전차로"}:
            table = rear if name == "후미_전차로" else rear.loc[rear["lane_role"].eq("through_or_shared")]
            if name == "후미_기본_2DTTC":
                negated = as_negated(table, "min_ttc_2d_s", "min_pet_s", ttc_status="ttc_2d_status",
                                     types=["ttc_2d_conflict_type", "pet_conflict_type"], ttc_jitter="jitter_ttc_2d")
            else:
                negated = as_negated(table, "min_ttc_s", "min_pet_s", strict=name == "후미_기본_엄격",
                                     speed_cols=["follower_speed_at_min_ttc_mps", "follower_speed_at_min_pet_mps"],
                                     types=["ttc_conflict_type", "pet_conflict_type"])
        elif name in {"교차_M1", "교차_M1_엄격"}:
            table = cross.loc[cross["m1_left_turn_opposing_through"].astype(str).str.lower().eq("true")]
            negated = as_negated(table, "min_ttc_s", "pet_s", strict=name == "교차_M1_엄격")
        else:
            raise ValueError("06에 없는 층: " + str(name))
        result[name] = (table, negated)
    return result


def select_reviews(rear, cross, thresholds, k, seed, source05_id, strata_names=None):
    if isinstance(k, bool) or not isinstance(k, (int, np.integer)) or k < 1:
        raise ValueError("K는 1 이상의 정수여야 합니다.")
    rear = prepare_catalog(rear, "rear_end", source05_id)
    cross = prepare_catalog(cross, "crossing", source05_id)
    strata = make_strata(rear, cross, ["후미_기본", "교차_M1"] if strata_names is None else strata_names)
    samples, inventory = [], []
    for stratum, (table, negated) in strata.items():
        for metric, column in [("ttc", "x"), ("pet", "y")]:
            key = thresholds["유형"].eq(stratum) & thresholds["지점"].eq("ALL") & thresholds["지표"].eq(metric)
            rows = thresholds.loc[key]
            info = {"stratum": stratum, "site_scope": "ALL", "metric": metric, "threshold": np.nan,
                    "n_tail": 0, "n_extreme": 0, "n_remainder": 0, "n_random": 0}
            if len(rows) > 1:
                raise ValueError(f"06 임계값 중복: {stratum} {metric}")
            status_columns = [c for c in ["fit", "status", "threshold_status", "candidate_status", "상태"] if c in rows]
            skip_status = next((status for status in ["no_stable_region", "too_few", "too_few_for_stability"]
                                if len(rows) and rows[status_columns].astype(str).eq(status).any(axis=None)), None)
            if skip_status:
                print(f"건너뜀: {stratum} {metric} — {skip_status}; THRESHOLD_SET_BY_LAYER[{stratum!r}] 설정을 확인하세요 (stable/default12).")
                inventory.append({**info, "status": skip_status})
                continue
            u = pd.to_numeric(rows.iloc[0].get("threshold"), errors="coerce") if len(rows) else np.nan
            if not np.isfinite(u):
                print(f"건너뜀: {stratum} {metric} — threshold_unavailable; THRESHOLD_SET_BY_LAYER[{stratum!r}] 설정을 확인하세요 (stable/default12).")
                inventory.append({**info, "status": "threshold_unavailable"})
                continue
            values = negated[column].to_numpy(dtype=float)
            tail = table.iloc[np.flatnonzero(np.isfinite(values) & (values > u))].copy()
            if "n_exceed" in rows:
                expected_count = pd.to_numeric(rows.iloc[0]["n_exceed"], errors="coerce")
                if not np.isfinite(expected_count) or len(tail) != expected_count:
                    raise ValueError(f"06 꼬리 수 불일치: {stratum} {metric}, 06={expected_count}, 07={len(tail)}")
            tail["negated_value"] = values[np.isfinite(values) & (values > u)]
            tail = tail.sort_values(["negated_value", "catalog_row"], ascending=[False, True], kind="stable")
            extreme, remainder = tail.iloc[:k].copy(), tail.iloc[k:].copy()
            key_seed = int.from_bytes(hashlib.sha256(f"{seed}|{stratum}|{metric}".encode()).digest()[:8], "big")
            rng = np.random.default_rng(key_seed)
            random = remainder.iloc[np.sort(rng.choice(len(remainder), min(k, len(remainder)), replace=False))].copy()
            for selected, method in [(extreme, "극단"), (random, "무작위")]:
                selected["stratum"], selected["metric"], selected["selection"] = stratum, metric, method
                selected["site_scope"], selected["threshold"] = "ALL", float(u)
                selected["value_s"] = -selected["negated_value"]
                selected["metric_key"] = "ttc_2d" if stratum == "후미_기본_2DTTC" and metric == "ttc" else metric
                selected["review_id"] = selected["event_id"] + ":" + stratum + ":" + metric
                samples.append(selected)
            inventory.append({**info, "threshold": float(u), "n_tail": len(tail), "n_extreme": len(extreme),
                              "n_remainder": len(remainder), "n_random": len(random),
                              "status": "selected" if len(tail) else "no_exceedances"})
    columns = ["review_id", "event_id", "source_file", "file_stem", "site", "catalog_kind", "catalog_row",
               "stratum", "site_scope", "metric", "metric_key", "value_s", "negated_value", "threshold", "selection", "figure_file"]
    selected = pd.concat(samples, ignore_index=True) if samples else pd.DataFrame(columns=columns)
    if selected["review_id"].duplicated().any():
        raise ValueError("점검 행 식별자가 중복입니다.")
    selected["review_number"] = np.arange(1, len(selected) + 1)
    number_width = max(3, len(str(len(selected))))
    selected["figure_file"] = [
        f"figures/{r['review_number']:0{number_width}d}_{r['stratum']}_{r['metric']}_{r['selection']}_"
        + hashlib.sha256(r["review_id"].encode()).hexdigest()[:24] + ".png"
        for r in selected.to_dict("records")]
    return selected, pd.DataFrame(inventory)


def validate_final_tail_counts(sampling, thresholds, source05_id, source06_id):
    if source06_id != "20260924T191302Z_4491ed18":
        return
    if source05_id != "20260924T125033Z_ffaaff43":
        raise ValueError("최종 06의 원본 05 ID가 다릅니다.")
    expected = {("후미_기본", "default12"): {"ttc": 17418, "pet": 28288},
                ("교차_M1", "stable"): {"ttc": 115, "pet": 134}}
    for (name, choice), counts in expected.items():
        if name not in REVIEW_STRATA:
            continue
        provenance = globals().get("threshold_selection", {})
        effective = provenance.get("by_layer", {}).get(name, provenance).get("effective")
        if effective is None:
            mapping = globals().get("THRESHOLD_SET_BY_LAYER")
            effective = mapping.get(name, THRESHOLD_SET) if mapping is not None else THRESHOLD_SET
        if effective != choice:
            continue
        for metric, count in counts.items():
            row = sampling.loc[sampling["stratum"].eq(name) & sampling["metric"].eq(metric)]
            threshold = thresholds.loc[thresholds["유형"].eq(name) & thresholds["지점"].eq("ALL")
                                       & thresholds["지표"].eq(metric)]
            if len(row) != 1 or len(threshold) != 1:
                raise ValueError(f"최종 06 대조 필수 행 누락·중복: {name} {choice} {metric}")
            if (threshold.iloc[0].get("threshold_set") != choice
                    or row.iloc[0]["status"] != "selected" or row.iloc[0]["n_tail"] != count):
                raise ValueError(f"최종 06 기준 꼬리 수 불일치: {name} {choice} {metric}, 기대={count}")

## 5. 입력 확인과 표본 확정
아직 궤적은 읽지 않습니다. 층별 꼬리 수와 선정 수부터 확인합니다.

In [5]:
rear, cross, thresholds, manifest, affines = load_inputs()
selected, sampling = select_reviews(rear, cross, thresholds, K, RANDOM_SEED, SOURCE05_ID, REVIEW_STRATA)
validate_final_tail_counts(sampling, thresholds, SOURCE05_ID, SOURCE06_ID)
selection_config = review_config()
display(sampling)
print("점검 행:", len(selected), "| 필요한 궤적 파일:", selected["source_file"].nunique())

사용 임계값: default12 | univariate_gpd_default12.csv
사용 임계값: stable | univariate_gpd_stable.csv
06 문헌 절차 적용: 4초 OR → 유턴 제외=True → stop_line_hull_v2 | 지도 20260916T134547Z_047caf90


,stratum,site_scope,metric,threshold,n_tail,n_extreme,n_remainder,n_random,status
0,후미_기본,ALL,ttc,-2.113019,17418,30,17388,30,selected
1,후미_기본,ALL,pet,-0.911016,28288,30,28258,30,selected
2,교차_M1,ALL,ttc,-2.761550,115,30,85,30,selected
3,교차_M1,ALL,pet,-3.183659,134,30,104,30,selected


점검 행: 240 | 필요한 궤적 파일: 173


## 6. 원자료와 좌표 처리 함수
필요한 파일마다 CSV를 **한 번만** 청크로 읽고, 선정된 차량·드론의 전체 관측만 메모리에 남깁니다.
관측 공백·중복 시각·좌표 결측을 선으로 잇지 않습니다. 원래 Ortho 위치를 평활하지 않고 표시합니다.
진행 방향은 05와 같은 방법으로 위치의 작은 흔들림을 줄인 뒤 계산합니다. 정지 시 같은 연속 구간의
직전/직후 유효 방향을 쓰며, 구간 전체가 정지하면 방향 미확인으로 차체를 생략합니다.
방향 보완에 쓰는 1 km/h는 05와 동일한 프로젝트 선택입니다. Fonod p.20의 그림 가독성용 필터를
검증된 정지 기준으로 해석하지 않습니다.
길이·폭은 그 시각 원자료를 쓰고 결측·0·음수이면 생략합니다. 차종·제공 속력은 원자료, km/h 단위입니다.
후미 극값의 05 계산 속력과 좌우 간격도 별도로 표시합니다.

차체의 실제 길이·폭·방향은 04의 좌표 변환을 거쳐 배경 영상의 픽셀 단위로 옮깁니다.
중심은 원래 Ortho 위치이며, 배경과 궤적·차체를 모두 같은 좌표계에 그립니다.

In [6]:
def event_actors(row):
    if row["catalog_kind"] == "rear_end":
        return [("앞차", str(row["leader_id"])), ("뒤차", str(row["follower_id"]))]
    return [("A", str(row["vehicle_a_id"])), ("B", str(row["vehicle_b_id"]))]


def event_times(row):
    if row["catalog_kind"] == "crossing" and row["metric"] == "pet":
        result = [("첫 통과", row.get("t_first_pass_s")), ("두 번째 통과", row.get("t_second_pass_s"))]
    else:
        result = [("극값", row.get("min_" + row.get("metric_key", row["metric"]) + "_time_s"))]
    result = [(label, float(t)) for label, t in result if pd.notna(t) and np.isfinite(float(t))]
    if not result:
        raise ValueError("극값 또는 통과 시각 미확인: " + row["review_id"])
    return result


def affine_matrix(row):
    a = np.array([[row["hand_a11"], row["hand_a12"]], [row["hand_a21"], row["hand_a22"]]], float)
    b = np.array([row["hand_b1"], row["hand_b2"]], float)
    if not np.isfinite(a).all() or not np.isfinite(b).all() or np.linalg.cond(a) > 1e8:
        raise ValueError("유효한 04 아핀 계수가 없습니다.")
    if row["status"] != "completed":
        raise ValueError("해당 파일의 04 처리가 완료되지 않았습니다.")
    if not np.isclose(np.linalg.det(a), float(row["hand_det"]), rtol=1e-5, atol=1e-12):
        raise ValueError("04 요약표와 아핀 계수의 행렬식이 다릅니다.")
    return a, b


def rectangle_pixels(center, heading, length, width, a):
    if not np.isfinite([*center, heading, length, width]).all() or length <= 0 or width <= 0:
        return None
    forward = np.array([np.cos(heading), np.sin(heading)])
    side = np.array([-forward[1], forward[0]])
    offsets = np.array([forward * x * length / 2 + side * y * width / 2
                        for x, y in [(1, 1), (1, -1), (-1, -1), (-1, 1)]])
    return np.asarray(center) + offsets @ np.linalg.inv(a).T


def smooth_positions(values):
    pad = min(len(values) - 1, int(4 * SIGMA_FRAMES))
    if pad < 1:
        return values.copy()
    extended = np.r_[2 * values[0] - values[pad:0:-1], values,
                     2 * values[-1] - values[-2:-pad - 2:-1]]
    # scipy gaussian_filter1d(sigma=14, truncate=4, mode='nearest')의 동일 이산 커널.
    radius = int(4 * SIGMA_FRAMES + .5)
    offsets = np.arange(-radius, radius + 1, dtype=float)
    weights = np.exp(-.5 * (offsets / SIGMA_FRAMES) ** 2)
    weights /= weights.sum()
    filtered = np.convolve(np.pad(extended, radius, mode="edge"), weights, mode="valid")
    return filtered[pad:pad + len(values)]


def prepare_actor(frame):
    frame = frame.copy()
    valid_clock = frame["Local_Time"].str.fullmatch(r"(?:[01]\d|2[0-3]):[0-5]\d:[0-5]\d(?:\.\d{1,9})?", na=False)
    frame["t"] = pd.to_timedelta(frame["Local_Time"].where(valid_clock), errors="coerce").dt.total_seconds()
    numeric = ["Ortho_X", "Ortho_Y", "Local_X", "Local_Y", "Vehicle_Length", "Vehicle_Width", "Vehicle_Class", "Vehicle_Speed"]
    for c in numeric:
        frame[c] = pd.to_numeric(frame[c], errors="coerce").astype("float64")
    duplicate = frame["t"].duplicated(keep=False)
    frame["valid"] = ~duplicate & np.isfinite(frame[["t", "Ortho_X", "Ortho_Y", "Local_X", "Local_Y"]]).all(axis=1)
    frame = frame.sort_values("t", kind="stable").reset_index(drop=True)
    breaks = (~frame["valid"] | ~frame["valid"].shift(1, fill_value=False)
              | frame["t"].diff().gt(NEXT_FRAME_MAX_S) | frame["t"].diff().le(0))
    frame["segment"] = breaks.cumsum()
    frame["heading"] = np.nan
    frame["smooth_speed_mps"] = np.nan
    frame["heading_filled_stationary"] = False
    for _, g in frame.loc[frame["valid"]].groupby("segment"):
        if len(g) < 2:
            continue
        vx = np.gradient(smooth_positions(g["Local_X"].to_numpy(float))) / FRAME_S
        vy = np.gradient(smooth_positions(g["Local_Y"].to_numpy(float))) / FRAME_S
        speed = np.hypot(vx, vy)
        heading = pd.Series(np.where(speed >= STATIONARY_MPS, np.arctan2(vy, vx), np.nan)).ffill().bfill()
        frame.loc[g.index, "heading"] = heading.to_numpy()
        frame.loc[g.index, "smooth_speed_mps"] = speed
        frame.loc[g.index, "heading_filled_stationary"] = (speed < STATIONARY_MPS) & heading.notna().to_numpy()
    return frame


def read_selected_file(path, expected, rows):
    path = within(path, RAW)
    assert_signature(path, expected)
    wanted = {(str(r["drone_id"]), actor) for r in rows.to_dict("records") for _, actor in event_actors(r)}
    columns = ["Vehicle_ID", "Drone_ID", "Local_Time", "Ortho_X", "Ortho_Y", "Local_X", "Local_Y",
               "Vehicle_Length", "Vehicle_Width", "Vehicle_Class", "Vehicle_Speed", "Road_Section", "Lane_Number"]
    parts = []
    with pd.read_csv(path, usecols=columns, dtype="string", chunksize=200000) as reader:
        for chunk in reader:
            keys = pd.MultiIndex.from_frame(chunk[["Drone_ID", "Vehicle_ID"]])
            part = chunk.loc[keys.isin(wanted)]
            if len(part):
                parts.append(part)
    assert_signature(path, expected)
    if not parts:
        raise ValueError("선정된 차량·드론이 원자료에 없습니다: " + str(path))
    raw = pd.concat(parts, ignore_index=True)
    return {(str(drone), str(actor)): prepare_actor(g)
            for (drone, actor), g in raw.groupby(["Drone_ID", "Vehicle_ID"], sort=False)}


def load_background(site):
    if not re.fullmatch(r"[A-Z]", str(site)):
        raise ValueError("지점 이름 오류")
    image_path = within(RAW / "orthophotos/orthophotos" / (site + ".png"), RAW)
    lanes_path = within(RAW / "segmentations/segmentations" / (site + ".csv"), RAW)
    before = signature(image_path)
    with Image.open(image_path) as original:
        background = original.convert("RGB")
    assert_signature(image_path, before)
    lane_sig = signature(lanes_path)
    lanes = pd.read_csv(lanes_path, dtype={"Section": "string", "Lane": "string"})
    assert_signature(lanes_path, lane_sig)
    fingerprints = [{"source_file": p.relative_to(PROJECT).as_posix(), **s}
                    for p, s in [(image_path, before), (lanes_path, lane_sig)]]
    return background, lanes, fingerprints

## 7. 그림과 점검표 함수
한 사건×지표당 PNG 하나를 만듭니다. 실선은 원래 관측 위치, 색 사각형은 표시 시각에 가장 가까운
관측(반 프레임+2ms 이내)의 치수·추정 방향입니다. 해당 관측과의 시간차를 패널에 기록합니다.
차로 경계는 원래 분할 다각형입니다. 범례의 속력은 제공값(km/h), 제목의 후미 속력은 05 계산값(m/s)입니다.
프레임·치수·방향 결측은 그림의 경고, 점검표 `요약`, 전체 명세표 `plot_notes`에 남깁니다.

In [7]:
def fmt(value, digits=3):
    try:
        return f"{float(value):.{digits}f}" if np.isfinite(float(value)) else "미확인"
    except (TypeError, ValueError):
        return "미확인"


def drawing_font(size):
    path = Path("C:/Windows/Fonts/malgun.ttf")
    if not path.is_file():
        raise FileNotFoundError("한글 글꼴 malgun.ttf를 찾지 못했습니다. drawing_font의 경로를 바꾸세요.")
    return ImageFont.truetype(str(path), size)


def check_drawing_environment():
    try:
        with Image.new("RGB", (240, 70), "white") as canvas:
            drawing = ImageDraw.Draw(canvas)
            for y, size in [(5, 17), (35, 14)]:
                drawing.text((5, y), "그림 환경 점검", font=drawing_font(size), fill="black",
                             stroke_width=1, stroke_fill="white")
            with io.BytesIO() as buffer:
                canvas.save(buffer, format="PNG")
                buffer.seek(0)
                with Image.open(buffer) as image:
                    image.verify()
    except Exception as error:
        raise RuntimeError("그림 환경 사전 점검에 실패했습니다. 한글 글꼴과 Pillow의 PNG 그리기·저장 "
                           f"환경을 확인하세요: {type(error).__name__}: {error}") from error


def draw_event(row, actors, background, lanes, affine, path, window_s):
    a, b = affine_matrix(affine)
    times = event_times(row)
    if not np.isfinite(window_s) or window_s <= 0:
        raise ValueError("WINDOW_S는 양수여야 합니다.")
    panel_w, panel_h, top, bottom = 850, 760, 180, 150
    canvas = Image.new("RGB", (panel_w * len(times), top + panel_h + bottom), "white")
    drawing = ImageDraw.Draw(canvas)
    font, small = drawing_font(17), drawing_font(14)
    notes, snapshot_rows, title_actors = [], [], []
    classes = {0: "승용·밴", 1: "버스", 2: "트럭", 3: "오토바이"}
    for panel_index, (time_label, t0) in enumerate(times):
        points, pieces, panel_notes = [], [], []
        for (label, actor), color in zip(event_actors(row), ["#00cfff", "#ff943b"]):
            track = actors.get((str(row["drone_id"]), actor))
            if track is None:
                panel_notes.append(label + " 궤적 없음")
                continue
            window = track.loc[track["t"].between(t0 - window_s, t0 + window_s)]
            xy_ok = np.isfinite(window[["Ortho_X", "Ortho_Y"]]).all(axis=1)
            xy = window.loc[xy_ok, ["Ortho_X", "Ortho_Y"]].to_numpy(float)
            if len(xy):
                points.append(xy)
            if not window["valid"].all():
                panel_notes.append(label + " 중복/좌표 결측: 선 연결 제외")
            valid = track.loc[track["valid"]]
            near = None if valid.empty else valid.loc[(valid["t"] - t0).abs().idxmin()]
            if near is None or abs(near["t"] - t0) > FRAME_S / 2 + .002:
                panel_notes.append(label + " 해당 시각 관측 없음: 차체 생략")
                near = None
            body = None
            if near is not None:
                body = rectangle_pixels(near[["Ortho_X", "Ortho_Y"]].to_numpy(float),
                                        near["heading"], near["Vehicle_Length"], near["Vehicle_Width"], a)
                if body is None:
                    panel_notes.append(label + " 방향/치수 미확인: 차체 생략")
                else:
                    points.append(body)
                snapshot_rows.append({"review_id": row["review_id"], "panel": time_label,
                                      "actor_id": actor, "requested_time_s": t0,
                                      "observed_time_s": float(near["t"]), "dt_s": float(near["t"] - t0),
                                      "class": near["Vehicle_Class"], "speed_kmh": near["Vehicle_Speed"],
                                      "length_m": near["Vehicle_Length"], "width_m": near["Vehicle_Width"],
                                      "heading_rad_local": near["heading"], "rectangle_drawn": body is not None,
                                      "smooth_speed_mps": near["smooth_speed_mps"],
                                      "heading_filled_stationary": bool(near["heading_filled_stationary"])})
                if near["heading_filled_stationary"]:
                    panel_notes.append(label + " 정지 중 방향 추정")
            if valid.empty or valid["t"].min() > t0 - window_s or valid["t"].max() < t0 + window_s:
                panel_notes.append(label + " ±창 일부 미관측")
            car_class = classes.get(near["Vehicle_Class"], "미확인") if near is not None else "미확인"
            speed = fmt(near["Vehicle_Speed"], 1) if near is not None else "미확인"
            caption = f"{label} ID={actor} {car_class}, 제공 {speed} km/h"
            if near is not None:
                caption += f" Δt={near['t'] - t0:+.3f}s"
                if near["heading_filled_stationary"]:
                    caption += " | 정지 중 방향 추정"
            title_actors.append(caption)
            pieces.append((label, actor, color, window, near, body, caption))
        if not points:
            raise ValueError("±시간 창에 원래 픽셀 위치가 없습니다: " + row["review_id"])
        all_xy = np.vstack(points)
        lo, hi = all_xy.min(axis=0) - 70, all_xy.max(axis=0) + 70
        x0, y0 = np.maximum(np.floor(lo), [0, 0]).astype(int)
        x1, y1 = np.minimum(np.ceil(hi), background.size).astype(int)
        if x1 <= x0 or y1 <= y0:
            raise ValueError("관측 위치가 정사영상 범위 밖입니다.")
        if np.any(all_xy < 0) or np.any(all_xy >= np.array(background.size)):
            panel_notes.append("일부 원래 위치가 영상 범위 밖")
        crop = background.crop((x0, y0, x1, y1))
        scale = min((panel_w - 70) / crop.width, (panel_h - 100) / crop.height)
        size = (max(1, round(crop.width * scale)), max(1, round(crop.height * scale)))
        crop = crop.resize(size, Image.Resampling.LANCZOS)
        overlay = ImageDraw.Draw(crop)
        # 실제 resize 비율을 적용해 배경과 도형의 반올림 오차를 방지합니다.
        def screen(xy):
            return ((np.asarray(xy, float) - [x0, y0] + .5) * np.array(size) / [x1 - x0, y1 - y0] - .5)
        for lane in lanes.to_dict("records"):
            polygon = np.array([[lane[x], lane[y]] for x, y in CORNERS], float)
            if not np.isfinite(polygon).all():
                raise ValueError("차로 다각형 좌표 결측")
            if np.any(polygon.max(axis=0) < [x0, y0]) or np.any(polygon.min(axis=0) > [x1, y1]):
                continue
            poly = [tuple(p) for p in screen(polygon)]
            overlay.line(poly + [poly[0]], fill="#c9ec77", width=1)
            center = screen(polygon.mean(axis=0))
            if 0 <= center[0] < size[0] and 0 <= center[1] < size[1]:
                overlay.text(tuple(center), f"{lane['Section']}:{lane['Lane']}", font=small,
                             fill="#eeffa9", stroke_width=1, stroke_fill="black")
        for label, actor, color, window, near, body, caption in pieces:
            for _, segment in window.loc[window["valid"]].groupby("segment"):
                xy = screen(segment[["Ortho_X", "Ortho_Y"]].to_numpy(float))
                coords = [tuple(p) for p in xy]
                if len(coords) > 1:
                    overlay.line(coords, fill=color, width=3)
                for x, y in coords:
                    overlay.ellipse((x-1, y-1, x+1, y+1), fill=color)
                if len(coords):
                    start_text = f"{label} 시작"
                    text_width = small.getlength(start_text)
                    text_xy = (min(max(0, coords[0][0]), max(0, size[0] - text_width - 2)),
                               min(max(0, coords[0][1]), max(0, size[1] - 20)))
                    overlay.text(text_xy, start_text, font=small, fill=color,
                                 stroke_width=1, stroke_fill="black")
            bad = window.loc[~window["valid"]].dropna(subset=["Ortho_X", "Ortho_Y"])
            for x, y in screen(bad[["Ortho_X", "Ortho_Y"]].to_numpy(float)):
                overlay.line((x-3, y-3, x+3, y+3), fill=color, width=2)
                overlay.line((x-3, y+3, x+3, y-3), fill=color, width=2)
            if body is not None:
                poly = [tuple(p) for p in screen(body)]
                overlay.line(poly + [poly[0]], fill="white", width=5)
                overlay.line(poly + [poly[0]], fill=color, width=3)
                # 앞면 중앙을 표시해 직사각형의 진행 방향을 구별합니다.
                nose = screen((body[0] + body[1]) / 2)
                center = screen(near[["Ortho_X", "Ortho_Y"]].to_numpy(float))
                overlay.line([tuple(center), tuple(nose)], fill=color, width=3)
            if near is not None:
                x, y = screen(near[["Ortho_X", "Ortho_Y"]].to_numpy(float))
                overlay.ellipse((x-4, y-4, x+4, y+4), fill=color, outline="black", width=1)
        if row["catalog_kind"] == "crossing":
            local_point = np.array([row.get("conflict_x", np.nan), row.get("conflict_y", np.nan)], float)
            if np.isfinite(local_point).all():
                x, y = screen(np.linalg.solve(a, local_point - b))
                overlay.line((x-7, y, x+7, y), fill="magenta", width=3)
                overlay.line((x, y-7, x, y+7), fill="magenta", width=3)
        left = panel_index * panel_w + (panel_w - size[0]) // 2
        upper = top + 65
        canvas.paste(crop, (left, upper))
        px = panel_index * panel_w + 16
        drawing.text((px, top), f"{time_label} {t0:.3f}s ±{window_s:g}s (자정 기준)", font=font, fill="black")
        for j, piece in enumerate(pieces):
            drawing.text((px, top + 23 + j * 19), piece[-1], font=small, fill="black")
            drawing.rectangle((px - 9, top + 27 + j * 19, px - 3, top + 37 + j * 19), fill=piece[2])
        drawing.text((left, upper + size[1] + 5),
                     f"Ortho X: {x0}–{x1} px | Y: {y0}–{y1} px (아래로 증가)", font=small, fill="black")
        for j, note in enumerate(panel_notes):
            drawing.text((px, top + panel_h + j * 19), note, font=small, fill="#9a2020")
        notes.extend(time_label + ": " + n for n in panel_notes)
    metric_key = row.get("metric_key", row["metric"])
    conflict = row.get(metric_key + "_conflict_type", "") if row["catalog_kind"] == "rear_end" else row.get("crossing_class", "")
    titles = [row["event_id"],
              f"{row['stratum']} ALL | {row['metric'].upper()}={fmt(row['value_s'])} s | {row['selection']} | 유형={conflict}"]
    if row["catalog_kind"] == "rear_end":
        titles.append(f"05 속력(앞/뒤)={fmt(row.get('leader_speed_at_min_' + metric_key + '_mps'))}/"
                      f"{fmt(row.get('follower_speed_at_min_' + metric_key + '_mps'))} m/s | "
                      f"좌우 간격={fmt(row.get('lateral_at_min_' + metric_key + '_m'))} m")
        if metric_key == "ttc_2d":
            notes.append("2D TTC 극값의 05 속력·좌우 간격 열 없음: 미확인; 차량별 제공 속력은 원자료 표시")
    else:
        titles.append("좌우 간격: 해당 없음(교차) | 속력: 표시 관측의 원자료 제공값")
    titles.extend(dict.fromkeys(title_actors))
    titles.append("원래 Ortho 궤적·차체 중심 | 방향: 평활 Local에서 추정 | 연두: 차로 | 분홍 십자: 교점")
    for j, title in enumerate(titles):
        drawing.text((16, 8 + j * 21), title, font=font if j < 2 else small, fill="black")
    canvas.save(target_path(path), format="PNG")
    canvas.close()
    return "; ".join(notes), snapshot_rows


def selected_sources(selected, manifest, affines):
    found = {}
    for source_file in selected["source_file"].unique():
        if selected.loc[selected["source_file"].eq(source_file), "site"].nunique(dropna=False) != 1:
            raise ValueError("동일 원자료 파일에 서로 다른 지점이 있습니다: " + source_file)
        expected = manifest.loc[manifest["source_file"].eq(source_file)]
        affine = affines.loc[affines["source_file"].eq(source_file)]
        if len(expected) != 1 or len(affine) != 1:
            raise ValueError("원자료 목록 또는 아핀의 단일 대응 없음: " + source_file)
        path = within(PROJECT / source_file, RAW)
        assert_signature(path, expected.iloc[0])
        affine_matrix(affine.iloc[0])
        found[source_file] = (path, expected.iloc[0], affine.iloc[0])
    return found


def render_reviews(selected, manifest, affines, window_s=3.0, sampling=None):
    sources = selected_sources(selected, manifest, affines)
    if (RUN_DIR / "review_sheet.csv").exists() or (RUN_DIR / "sample_manifest.csv").exists():
        raise FileExistsError("점검표 덮어쓰기를 막았습니다. 새 실행 폴더를 만드세요.")
    check_drawing_environment()
    (target_path(RUN_DIR / "figures")).mkdir(exist_ok=True)
    total_bytes = sum(int(s[1]["size_bytes"]) for s in sources.values())
    estimate = total_bytes / 1e6 / EST_READ_MBPS + len(selected) * EST_FIGURE_S
    print(f"예상 약 {estimate / 60:.1f}분 (가정 {EST_READ_MBPS:g} MB/s, 그림당 {EST_FIGURE_S:g}초; 환경별 차이)")
    started = time.perf_counter()
    result = selected.copy()
    result["plot_notes"] = ""
    result["figure_status"] = "pending"
    snapshots, signatures, reads = [], [], []
    processed_bytes, processed_figures = 0, 0
    read_seconds, plot_seconds = 0.0, 0.0
    for site, site_rows in selected.groupby("site", sort=True):
        background, lanes, background_signatures = load_background(site)
        signatures.extend(background_signatures)
        try:
            for source_file, rows in site_rows.groupby("source_file", sort=True):
                path, expected, affine = sources[source_file]
                t = time.perf_counter()
                actors = read_selected_file(path, expected, rows)
                read_seconds += time.perf_counter() - t
                processed_bytes += int(expected["size_bytes"])
                signatures.append({"source_file": source_file, **signature(path)})
                reads.append({"source_file": source_file, "csv_read_calls": 1, "n_review_rows": len(rows)})
                t = time.perf_counter()
                for index, row in rows.iterrows():
                    try:
                        notes, observed = draw_event(row, actors, background, lanes, affine,
                                                     RUN_DIR / row["figure_file"], window_s)
                    except Exception as error:
                        result.loc[index, "figure_status"] = "error"
                        result.loc[index, "plot_notes"] = f"그림 생성 오류: {type(error).__name__}: {error}"
                        print("그림 생성 오류, 다음 사건 계속:", row["review_id"], "|", error)
                    else:
                        result.loc[index, "plot_notes"] = notes
                        result.loc[index, "figure_status"] = "with_notes" if notes else "ok"
                        snapshots.extend(observed)
                plot_seconds += time.perf_counter() - t
                processed_figures += len(rows)
                remaining = ((total_bytes - processed_bytes) * read_seconds / max(processed_bytes, 1)
                             + (len(selected) - processed_figures) * plot_seconds / processed_figures)
                print(f"{len(reads)}/{len(sources)} 파일 | {processed_figures}/{len(selected)} 그림"
                      f" | 경과 {(time.perf_counter() - started) / 60:.1f}분 | 남은 예상 {remaining / 60:.1f}분")
                del actors
        finally:
            background.close()
    save_csv(result, RUN_DIR / "sample_manifest.csv")
    sheet = make_review_sheet(result, sampling)
    sheet.attrs["n_figure_errors"] = int(result["figure_status"].eq("error").sum())
    sheet.attrs["n_figures_successful"] = int(result["figure_status"].isin(["ok", "with_notes"]).sum())
    save_csv(sheet, RUN_DIR / "review_sheet.csv")
    save_csv(pd.DataFrame(snapshots, columns=["review_id", "panel", "actor_id", "requested_time_s",
             "observed_time_s", "dt_s", "class", "speed_kmh", "length_m", "width_m",
             "heading_rad_local", "rectangle_drawn", "smooth_speed_mps", "heading_filled_stationary"]),
             RUN_DIR / "displayed_observations.csv")
    save_csv(pd.DataFrame(signatures, columns=["source_file", "size_bytes", "mtime_ns"]), RUN_DIR / "input_signatures.csv")
    save_csv(pd.DataFrame(reads, columns=["source_file", "csv_read_calls", "n_review_rows"]), RUN_DIR / "raw_read_counts.csv")
    return sheet


def make_review_sheet(result, sampling=None):
    pet_comparison = "정의 동일, 임계값·꼬리 수 비교 미확인"
    if sampling is not None:
        pet = sampling.loc[sampling["metric"].eq("pet")]
        base = pet.loc[pet["stratum"].eq("후미_기본")]
        two_d = pet.loc[pet["stratum"].eq("후미_기본_2DTTC")]
        if len(base) == len(two_d) == 1:
            comparison = pd.concat([base, two_d])[["threshold", "n_tail"]].apply(pd.to_numeric, errors="coerce")
            if np.isfinite(comparison.to_numpy()).all():
                same = comparison.iloc[0].eq(comparison.iloc[1]).all()
                pet_comparison = "정의·꼬리 동일(중복 점검)" if same else "정의 동일, 임계값 다름"
    sheet = pd.DataFrame({"번호": result["review_number"], "review_id": result["review_id"],
                          "그림 파일": result["figure_file"], "층": result["stratum"],
                          "지표": result["metric"], "선정 방식": result["selection"],
                          "값(s)": pd.to_numeric(result["value_s"], errors="coerce").round(6)})
    notes = []
    for row in result.to_dict("records"):
        note = f"{row['site']} / {row['file_stem']} | 그림={row['figure_status']}"
        if row["stratum"] == "후미_기본_2DTTC" and row["metric"] == "pet":
            note += " | 후미_기본 PET와 " + pet_comparison
        if row["plot_notes"]:
            note += " | " + row["plot_notes"]
        notes.append(note)
    sheet["요약"] = notes
    for field in REVIEW_FIELDS:
        sheet[field] = ""
    return sheet

## 8. 새 실행 폴더
선정된 원자료 서명을 검사한 다음 새 RUN_ID를 만듭니다. 이전 점검표를 수정하거나 덮어쓰지 않습니다.
그림을 다시 만들려면 이 셀부터 새 실행을 시작합니다. 입력 변경 후에는 앞의 입력·선정 셀부터 다시 실행합니다.

In [8]:
if selection_config != review_config():
    raise ValueError("선정 뒤 입력/표본 설정이 바뀌었습니다. 5번부터 다시 실행하세요.")
selected_sources(selected, manifest, affines)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RUN_DIR = within(OUTPUT_ROOT / RUN_ID, OUTPUT_ROOT)
target_path(RUN_DIR / "run_metadata.json")
RUN_DIR.mkdir(parents=True, exist_ok=False)
run_metadata = {"run_id": RUN_ID, "source04_run_id": SOURCE04_ID, "source05_run_id": SOURCE05_ID,
                "source06_run_id": SOURCE06_ID, "started_at_utc": now_utc(), "status": "ready",
                "k": K, "seed": RANDOM_SEED, "window_s": WINDOW_S, "n_review_rows": len(selected),
                "threshold_selection": threshold_selection, "review_strata": list(REVIEW_STRATA),
                "literature_procedure": literature_procedure,
                "source06_function_sha256": SOURCE06_FUNCTION_SHA256,
                "as_negated_sha256": NEGATION_SHA256, "source_control_fingerprints": controls,
                "coordinate_system": "original Ortho pixels; Local metric vectors transformed by inverse 04 affine",
                "sampling": "top K then simple random without replacement from remainder",
                "validation": "human trajectory plausibility review; no crash prediction or accuracy claim"}
save_json(run_metadata, RUN_DIR / "run_metadata.json")
save_csv(sampling, RUN_DIR / "sampling_counts.csv")
if not literature_audit.empty:
    save_csv(literature_audit, RUN_DIR / "literature_prescreen_counts.csv")
print("이번 결과 폴더:", RUN_DIR)

이번 결과 폴더: C:\Users\123\Documents\(송도) 교통 연구 논문\data\processed\songdo_tail_review\20260924T201425Z_865a93dc


## 9. 필요한 파일만 읽어 그림과 빈 점검표 저장
먼저 한글 글꼴과 PNG 그리기 환경을 시험하고, 실패하면 원자료를 읽기 전에 멈춥니다.
파일별로 모아 한 번씩 읽고 그림을 저장하며 실행 시간 예상을 갱신합니다.
`review_sheet.csv`에서 **판정**은 오류 / 의심 / 정상, **원인**은 위치 / 방향 / 속력 / 크기 /
추적불일치 / 분리검출 / 기타입니다. 원인이 여러 개면 `;`, `,`, `，`로 구분합니다. 앞뒤 공백과 빈 조각은 무시합니다.
메모는 자유롭게 씁니다. 엑셀에서는 **'CSV UTF-8'로 저장**하세요.
번호 순서로 그림을 보고 판정합니다. `값(s)`는 소수 6자리 표시용이며 원래 값은 `sample_manifest.csv`에 있습니다.
그림 생성이 실패한 사건은 요약 칸에 오류를 남기고 다음 사건을 계속 그립니다. 이는 사람의 '오류' 판정이 아니며,
실패 행은 판정칸을 비워 두고 원인을 확인하세요. 입력 파일·서명 오류는 전체 실행을 중단합니다.
화면상의 한 장면으로 확정하기 어려우면 의심으로 남깁니다. 같은 사건의 TTC·PET 판정은 각각의 영향을 점검합니다.

In [9]:
if run_metadata["status"] != "ready":
    raise RuntimeError("이미 실행한 셀입니다. 새 실행 폴더를 만드세요.")
run_metadata["status"] = "rendering"
save_json(run_metadata, RUN_DIR / "run_metadata.json")
try:
    review_sheet = render_reviews(selected, manifest, affines, WINDOW_S, sampling=sampling)
except BaseException as error:
    run_metadata.update(status="incomplete", error=repr(error), stopped_at_utc=now_utc())
    save_json(run_metadata, RUN_DIR / "run_metadata.json")
    raise
run_metadata.update(status="awaiting_review", figures_finished_at_utc=now_utc(),
                    n_figure_errors=review_sheet.attrs["n_figure_errors"])
save_json(run_metadata, RUN_DIR / "run_metadata.json")
print("그림:", RUN_DIR / "figures", "\n점검표:", RUN_DIR / "review_sheet.csv")
display(review_sheet.head())
print(f"그림 성공 {review_sheet.attrs['n_figures_successful']}장 / 오류 {run_metadata['n_figure_errors']}건")

예상 약 23.3분 (가정 25 MB/s, 그림당 2초; 환경별 차이)
1/173 파일 | 1/240 그림 | 경과 0.0분 | 남은 예상 5.0분
2/173 파일 | 2/240 그림 | 경과 0.1분 | 남은 예상 4.9분
3/173 파일 | 3/240 그림 | 경과 0.1분 | 남은 예상 5.1분
4/173 파일 | 4/240 그림 | 경과 0.1분 | 남은 예상 4.7분
5/173 파일 | 5/240 그림 | 경과 0.1분 | 남은 예상 4.5분
6/173 파일 | 6/240 그림 | 경과 0.1분 | 남은 예상 4.5분
7/173 파일 | 7/240 그림 | 경과 0.2분 | 남은 예상 4.4분
8/173 파일 | 9/240 그림 | 경과 0.2분 | 남은 예상 4.3분
9/173 파일 | 10/240 그림 | 경과 0.2분 | 남은 예상 4.3분
10/173 파일 | 11/240 그림 | 경과 0.3분 | 남은 예상 4.2분
11/173 파일 | 12/240 그림 | 경과 0.3분 | 남은 예상 4.2분
12/173 파일 | 13/240 그림 | 경과 0.3분 | 남은 예상 4.1분
13/173 파일 | 14/240 그림 | 경과 0.3분 | 남은 예상 4.1분
14/173 파일 | 15/240 그림 | 경과 0.3분 | 남은 예상 4.1분
15/173 파일 | 17/240 그림 | 경과 0.4분 | 남은 예상 4.0분
16/173 파일 | 19/240 그림 | 경과 0.4분 | 남은 예상 4.0분
17/173 파일 | 20/240 그림 | 경과 0.4분 | 남은 예상 4.0분
18/173 파일 | 22/240 그림 | 경과 0.5분 | 남은 예상 4.0분
19/173 파일 | 23/240 그림 | 경과 0.5분 | 남은 예상 3.9분
20/173 파일 | 24/240 그림 | 경과 0.5분 | 남은 예상 3.9분
21/173 파일 | 27/240 그림 | 경과 0.5분 | 남은 예상 3.9분
22/173 파일 | 30/240 그림 | 경과 0.6분 

,번호,review_id,그림 파일,층,지표,선정 방식,값(s),요약,판정,원인,메모
0,1,20260924T125033Z_ffaaff43:rear_end:00451977:후미...,figures/001_후미_기본_ttc_극단_19c2da530eb1e75a821a8...,후미_기본,ttc,극단,0.000238,S / 2022-10-07_S_AM5 | 그림=with_notes | 극값: 앞차 ...,,,
1,2,20260924T125033Z_ffaaff43:rear_end:00052514:후미...,figures/002_후미_기본_ttc_극단_9f7efb978eb7f7b4d739b...,후미_기본,ttc,극단,0.004604,L / 2022-10-04_L_AM4 | 그림=with_notes | 극값: 뒤차 ...,,,
2,3,20260924T125033Z_ffaaff43:rear_end:00321064:후미...,figures/003_후미_기본_ttc_극단_a0975de979c4fdd8f6116...,후미_기본,ttc,극단,0.007565,S / 2022-10-06_S_AM4 | 그림=with_notes | 극값: 앞차 ...,,,
3,4,20260924T125033Z_ffaaff43:rear_end:00409636:후미...,figures/004_후미_기본_ttc_극단_84ab344d4d865fde4bfbf...,후미_기본,ttc,극단,0.014032,L / 2022-10-07_L_AM2 | 그림=with_notes | 극값: 앞차 ...,,,
4,5,20260924T125033Z_ffaaff43:rear_end:00052515:후미...,figures/005_후미_기본_ttc_극단_5146f945e2e39c88c199e...,후미_기본,ttc,극단,0.014642,L / 2022-10-04_L_AM4 | 그림=with_notes | 극값: 뒤차 ...,,,


그림 성공 240장 / 오류 0건


## 10. 판정 검증과 비율 함수
모든 칸이 빈 행은 읽을 때 제외하고 개수를 알립니다. 그 외 행의 추가·삭제·중복 및
review_id·층·지표·선정 방식 변경은 거부합니다. 행 순서 변경은 허용합니다.
번호·그림 경로·표시값·요약의 저장 형식이 바뀌어도 원래 정보를 전체 명세표에서 다시 연결합니다.
미판정은 별도로 세며 분모는 **판정 완료 수**입니다.
무작위 표본을 전부 판정한 경우 오류율에 Wilson 이항 95% 구간을 보입니다.

**표본 설계에 맞는 해석:** 무작위 표본은 극단 K개를 뺀 나머지 꼬리의 표본입니다. 그 오류율만으로
전체 꼬리의 오류율이라고 부르지 않습니다. 극단군 N_E개를 모두 점검한 오류 수 e_E,
나머지 N_R개의 무작위 표본 오류율 p_R로 전체를 `(e_E + N_R × p_R)/(N_E+N_R)`로 추정합니다.
양쪽 표본을 모두 판정했을 때만 전체 추정과 나머지 Wilson 구간을 같은 식으로 변환한 95% 구간을 보입니다.
나머지를 전수 점검한 경우에는 전체 관측 비율과 점 구간을 보입니다.
이항 구간은 비복원 추출의 유한모집단 보정·사건 간 의존·사람 간 판정 불확실성을 반영하지 않는 근사입니다.
극단군과 무작위군의 단순 합산 비율은 전체 꼬리 추정으로 쓰지 않습니다.

In [10]:
def validate_review(sheet, baseline):
    required = {"review_id", *REVIEW_KEYS, *REVIEW_FIELDS}
    if len(sheet) != len(baseline):
        raise ValueError(f"점검표 행 수가 다릅니다: 기대 {len(baseline)}행 / 실제 {len(sheet)}행.")
    if not required.issubset(sheet.columns) or sheet.columns.duplicated().any():
        raise ValueError("점검표 열·행·식별자가 변경되었습니다.")
    sheet = sheet.copy()
    for column in required - {"메모"}:
        sheet[column] = sheet[column].fillna("").astype(str).str.strip()
    for column in ["판정", "원인"]:
        sheet[column] = sheet[column].str.normalize("NFC")
    ids = sheet["review_id"]
    if (ids.eq("").any() or ids.duplicated().any() or baseline["review_id"].duplicated().any()
            or set(ids) != set(baseline["review_id"])):
        raise ValueError("점검표 review_id 집합·행이 변경되었습니다.")
    original = baseline.set_index("review_id").loc[ids].reset_index()
    for human, source in REVIEW_KEYS.items():
        if not np.array_equal(sheet[human].to_numpy(), original[source].astype(str).to_numpy()):
            raise ValueError("점검표 핵심 열을 수정하지 마세요: " + human)
    allowed = {"", "오류", "의심", "정상"}
    if not sheet["판정"].isin(allowed).all():
        raise ValueError("판정은 빈 칸 / 오류 / 의심 / 정상 중 하나여야 합니다.")
    causes = {"위치", "방향", "속력", "크기", "추적불일치", "분리검출", "기타"}
    normalized_causes = []
    for text in sheet["원인"]:
        parts = [part.strip() for part in re.split(r"[;,，]", text) if part.strip()]
        if not set(parts).issubset(causes):
            raise ValueError("원인은 지정 항목을 ; 또는 , 또는 ，로 구분하세요: " + text)
        normalized_causes.append(";".join(parts))
    original["판정"] = sheet["판정"].to_numpy()
    original["원인"] = normalized_causes
    original["메모"] = sheet["메모"].fillna("").to_numpy()
    return original


def wilson(k, n):
    if n == 0:
        return np.nan, np.nan
    z = 1.959963984540054
    p = k / n
    denominator = 1 + z * z / n
    center = (p + z * z / (2 * n)) / denominator
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denominator
    return max(0., center - half), min(1., center + half)


def summarize_reviews(sheet, sampling):
    summaries, estimates = [], []
    for info in sampling.to_dict("records"):
        group = sheet.loc[sheet["stratum"].eq(info["stratum"]) & sheet["metric"].eq(info["metric"])]
        by_method = {}
        for method, planned in [("극단", int(info["n_extreme"])), ("무작위", int(info["n_random"]))]:
            part = group.loc[group["selection"].eq(method)]
            judged = part.loc[part["판정"].isin(["오류", "의심", "정상"])]
            counts = {label: int(judged["판정"].eq(label).sum()) for label in ["오류", "의심", "정상"]}
            n = len(judged)
            complete = len(part) == planned and n == planned
            low, high = wilson(counts["오류"], n) if method == "무작위" and complete else (np.nan, np.nan)
            summaries.append({"층": info["stratum"], "지표": info["metric"], "선정방식": method,
                              "선정수": planned, "판정완료": n, "미판정": planned - n, **counts,
                              **{label + "_비율": count / n if n else np.nan for label, count in counts.items()},
                              "오류_Wilson95_하한": low, "오류_Wilson95_상한": high,
                              "추정대상": "극단 제외 나머지 꼬리" if method == "무작위" else "선정 극단군"})
            by_method[method] = (complete, n, counts["오류"], low, high)
        ne, nr, total = int(info["n_extreme"]), int(info["n_remainder"]), int(info["n_tail"])
        e_complete, _, e_errors, _, _ = by_method["극단"]
        r_complete, rn, r_errors, rlow, rhigh = by_method["무작위"]
        estimate = low = high = np.nan
        status = "incomplete_review"
        if info["status"] in {"threshold_unavailable", "no_stable_region", "too_few", "too_few_for_stability"}:
            status = info["status"]
        elif total == 0:
            status = "no_exceedances"
        elif e_complete and (nr == 0 or (r_complete and rn > 0)):
            estimate = (e_errors + (nr * r_errors / rn if nr else 0)) / total
            if nr == 0 or rn == nr:
                low = high = estimate
                status = "census"
            else:
                low, high = (e_errors + nr * rlow) / total, (e_errors + nr * rhigh) / total
                status = "weighted_estimate"
        estimates.append({"층": info["stratum"], "지표": info["metric"], "꼬리전체수": total,
                          "극단전수수": ne, "나머지수": nr, "전체꼬리_오류율": estimate,
                          "전체꼬리_95하한": low, "전체꼬리_95상한": high, "상태": status})
    return pd.DataFrame(summaries), pd.DataFrame(estimates)

## 11. 점검표 작성 후 실행
새 커널에서도 아래 `REVIEW_RUN_ID`에 07 실행 ID를 넣어 요약할 수 있습니다. 준비·설정·함수 정의 셀만 먼저 실행하고,
입력 선정·새 실행·그림 실행 셀은 건너뛰세요. 요약은 동일 폴더에 저장하며 사람이 작성한 표는 덮어쓰지 않습니다.
결과는 자료 품질 점검이며 **정확도·사고 예측 성능·인과 효과의 근거가 아닙니다.**

In [12]:
REVIEW_RUN_ID = "20260924T201425Z_865a93dc"  # 재시작한 커널에서는 요약할 07 실행 ID를 입력
if REVIEW_RUN_ID:
    RUN_DIR = run_input("songdo_tail_review", REVIEW_RUN_ID)
elif "RUN_DIR" not in globals():
    raise ValueError("REVIEW_RUN_ID를 입력하세요.")
within(RUN_DIR, OUTPUT_ROOT)
run_metadata = read_control(RUN_DIR / "run_metadata.json")
SOURCE04_ID, SOURCE05_ID, SOURCE06_ID = [run_metadata[k] for k in ["source04_run_id", "source05_run_id", "source06_run_id"]]
SOURCE04 = run_input("songdo_structure", SOURCE04_ID)
SOURCE05 = run_input("songdo_events", SOURCE05_ID)
SOURCE06 = run_input("songdo_evt", SOURCE06_ID)
SOURCE01 = PROJECT / "data/processed/songdo_movement"
sheet = read_review_sheet(RUN_DIR / "review_sheet.csv")
baseline = read_control(RUN_DIR / "sample_manifest.csv", dtype=str, keep_default_na=False)
sampling = read_control(RUN_DIR / "sampling_counts.csv")
sheet = validate_review(sheet, baseline)
summary, tail_estimates = summarize_reviews(sheet, sampling)
display(summary)
display(tail_estimates)
save_csv(summary, RUN_DIR / "review_summary.csv")
save_csv(tail_estimates, RUN_DIR / "tail_error_estimates.csv")
run_metadata.update(status=("no_review_targets" if sheet.empty else
                            "reviewed" if sheet["판정"].ne("").all() else "partially_reviewed"),
                    summary_at_utc=now_utc(), reviewed_rows=int(sheet["판정"].ne("").sum()))
save_json(run_metadata, RUN_DIR / "run_metadata.json")
print("요약 저장:", RUN_DIR, "| 미판정", int(sheet["판정"].eq("").sum()))

점검표 읽기 인코딩: utf-8-sig


,층,지표,선정방식,선정수,판정완료,미판정,오류,의심,정상,오류_비율,의심_비율,정상_비율,오류_Wilson95_하한,오류_Wilson95_상한,추정대상
0,후미_기본,ttc,극단,30,30,0,20,10,0,0.666667,0.333333,0.000000,NaN,NaN,선정 극단군
1,후미_기본,ttc,무작위,30,30,0,0,2,28,0.000000,0.066667,0.933333,0.0,0.113513,극단 제외 나머지 꼬리
2,후미_기본,pet,극단,30,30,0,19,11,0,0.633333,0.366667,0.000000,NaN,NaN,선정 극단군
3,후미_기본,pet,무작위,30,30,0,0,17,13,0.000000,0.566667,0.433333,0.0,0.113513,극단 제외 나머지 꼬리
4,교차_M1,ttc,극단,30,30,0,0,23,7,0.000000,0.766667,0.233333,NaN,NaN,선정 극단군
5,교차_M1,ttc,무작위,30,30,0,0,28,2,0.000000,0.933333,0.066667,0.0,0.113513,극단 제외 나머지 꼬리
6,교차_M1,pet,극단,30,30,0,0,0,30,0.000000,0.000000,1.000000,NaN,NaN,선정 극단군
7,교차_M1,pet,무작위,30,30,0,0,0,30,0.000000,0.000000,1.000000,0.0,0.113513,극단 제외 나머지 꼬리


,층,지표,꼬리전체수,극단전수수,나머지수,전체꼬리_오류율,전체꼬리_95하한,전체꼬리_95상한,상태
0,후미_기본,ttc,17418,30,17388,0.001148,0.001148,0.114466,weighted_estimate
1,후미_기본,pet,28288,30,28258,0.000672,0.000672,0.114065,weighted_estimate
2,교차_M1,ttc,115,30,85,0.000000,0.000000,0.083901,weighted_estimate
3,교차_M1,pet,134,30,104,0.000000,0.000000,0.088100,weighted_estimate


요약 저장: C:\Users\123\Documents\(송도) 교통 연구 논문\data\processed\songdo_tail_review\20260924T201425Z_865a93dc | 미판정 0
